<div style="display:block;width:100%;margin:auto;" direction=rtl align=center>
    <br><br>
    <div style="width:100%;margin:100;display:block;background-color:#fff0;" display=block align=center>
        <table style="border-style:hidden;border-collapse:collapse;">
            <tr>
                <td style="border: none!important;">
                    <img width=130 align=right src="https://i.ibb.co/yXKQmtZ/logo1.png" style="margin:0;" />
                </td>
                <td style="text-align:center;border: none!important;">
                    <h1 align=center><font size=5 color="#045F5F"> <b> Large Language Models </b><br><br>Computer Assignment 1</font></h1>
                </td>
                <td style="border: none!important;">
                    <img width=170 align=left src="https://i.ibb.co/wLjqFkw/logo2.png" style="margin:0;" />
                </td>
            </tr>
        </table>
        <h1> Behzad Jannati / Sobhan Abedi - Final Project -
        <h1> 810103098 /810103081 </h1>
        <h1> Prof. Mohammad Javad Dousti & Yadoulah Yaghoubzadeh
    </div>
</div>

# Large Language Model Pruning Without Healing

# Part 2: Layer Pruning Implementation for Multiple Models

### Compatible with Google Colab T4 GPU (14GB VRAM)

>[Large Language Model Pruning Without Healing](#scrollTo=gTVXVnOguog4)

>[Part 2: Layer Pruning Implementation for Multiple Models](#scrollTo=gTVXVnOguog4)

>>>[Compatible with Google Colab T4 GPU (14GB VRAM)](#scrollTo=gTVXVnOguog4)

>>[Environment Setup & Imports](#scrollTo=ZrsTv9aVvjWO)

>>>[Import necessary libraries](#scrollTo=OI2dEg01uK0K)

>>>[GPU Memory Management Utilities](#scrollTo=3YHY7jK2wDKB)

>>[Model Loading with 8-bit Quantization](#scrollTo=ePLrh9KHwhZh)

>>[Layer Pruning Implementation](#scrollTo=Qbye6QyjxJbE)

>>[Evaluation Functions](#scrollTo=R2lfg5LpxZ8L)

>>[Main Pruning Pipeline](#scrollTo=F15WfwMnxkZi)

>>[Results Visualization](#scrollTo=Cgt5Y6hhx33c)

>>[Main Execution](#scrollTo=lVXdttHKyDSJ)





 This notebook implements layer pruning without healing for various LLMs using 8-bit quantization to fit within T4 GPU constraints.

## 1. Environment Setup & Imports

In [4]:
# Install required packages (if not already installed)
!pip install transformers>=4.36.0
!pip install bitsandbytes>=0.41.0
!pip install accelerate>=0.25.0
!pip install datasets>=2.14.0
!pip install sentencepiece==0.1.99
!pip install protobuf==3.20.3
!pip install torch>=2.0.0

In [5]:
!git config --global credential.helper store

In [ ]:
!hf auth login --token LLM_token --add-to-git-credential

Token is valid (permission: fineGrained).
The token `llm` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `llm`


### Import necessary libraries

In [10]:
import os
import gc
import json
import torch
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Transformers imports
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)
from datasets import load_dataset

In [11]:
# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

### 2. GPU Memory Management Utilities

In [12]:
def clear_gpu_memory():
    """Clear GPU memory cache"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print(f"GPU memory cleared. Available: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f}GB")

def get_gpu_memory_info():
    """Get current GPU memory usage"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        free = (torch.cuda.get_device_properties(0).total_memory / 1024**3) - reserved
        print(f"GPU Memory - Allocated: {allocated:.2f}GB, Reserved: {reserved:.2f}GB, Free: {free:.2f}GB")

## 3. Model Loading with 8-bit Quantization

In [13]:
def load_model_8bit(model_name, device="cuda"):
    """Load model with 8-bit quantization to save memory"""
    print(f"Loading {model_name} with 8-bit quantization...")

    # Configure 8-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.float16,
        bnb_8bit_quant_type="nf4",
        bnb_8bit_use_double_quant=True,
    )

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load model with 8-bit quantization
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
    )

    model.eval()
    get_gpu_memory_info()

    return model, tokenizer

## 4. Layer Pruning Implementation

In [14]:
class LayerPruner:
    """Handles layer pruning operations for transformer models"""

    def __init__(self, model, model_name):
        self.model = model
        self.model_name = model_name
        self.original_layers = self._get_model_layers()
        self.num_layers = len(self.original_layers)

    def _get_model_layers(self):
        """Get transformer layers from model"""
        # Handle different model architectures
        if hasattr(self.model, 'model') and hasattr(self.model.model, 'layers'):
            return self.model.model.layers
        elif hasattr(self.model, 'transformer') and hasattr(self.model.transformer, 'h'):
            return self.model.transformer.h
        elif hasattr(self.model, 'gpt_neox') and hasattr(self.model.gpt_neox, 'layers'):
            return self.model.gpt_neox.layers
        else:
            raise ValueError(f"Unknown model architecture for {self.model_name}")

    def prune_layers(self, start_idx, end_idx):
        """Remove layers from start_idx to end_idx (inclusive)"""
        if start_idx < 0 or end_idx >= self.num_layers or start_idx > end_idx:
            raise ValueError(f"Invalid layer indices: {start_idx} to {end_idx}")

        # Create new layer list excluding pruned layers
        new_layers = []
        for i in range(self.num_layers):
            if i < start_idx or i > end_idx:
                new_layers.append(self.original_layers[i])

        # Update model layers
        if hasattr(self.model, 'model') and hasattr(self.model.model, 'layers'):
            self.model.model.layers = torch.nn.ModuleList(new_layers)
        elif hasattr(self.model, 'transformer') and hasattr(self.model.transformer, 'h'):
            self.model.transformer.h = torch.nn.ModuleList(new_layers)
        elif hasattr(self.model, 'gpt_neox') and hasattr(self.model.gpt_neox, 'layers'):
            self.model.gpt_neox.layers = torch.nn.ModuleList(new_layers)

        print(f"Pruned layers {start_idx} to {end_idx} ({end_idx-start_idx+1} layers removed)")

    def get_optimal_pruning_layers(self, pruning_percentage):
        """Get optimal layers to prune based on similarity analysis results"""
        # Based on the analysis from part 1, we use the deeper layers strategy
        num_to_prune = int(self.num_layers * pruning_percentage / 100)
        if num_to_prune == 0:
            return None, None

        # Prune from deeper layers (excluding the last layer)
        end_idx = self.num_layers - 2  # Don't prune the last layer
        start_idx = max(0, end_idx - num_to_prune + 1)

        return start_idx, end_idx

## 5. Evaluation Functions

In [15]:
def evaluate_perplexity(model, tokenizer, dataset, num_samples=50, max_length=512):
    """Evaluate model perplexity on dataset"""
    model.eval()
    total_loss = 0
    total_tokens = 0

    with torch.no_grad():
        for i in tqdm(range(min(num_samples, len(dataset))), desc="Calculating perplexity"):
            text = dataset[i]['text']

            # Tokenize with proper handling for 8-bit models
            inputs = tokenizer(
                text,
                return_tensors="pt",
                max_length=max_length,
                truncation=True,
                padding=False
            )

            # Ensure input_ids are long type
            input_ids = inputs.input_ids.long().to(model.device)

            if input_ids.shape[1] < 2:
                continue

            # Calculate loss
            outputs = model(input_ids, labels=input_ids)
            total_loss += outputs.loss.item() * input_ids.shape[1]
            total_tokens += input_ids.shape[1]

    perplexity = torch.exp(torch.tensor(total_loss / total_tokens))
    return perplexity.item()

def evaluate_generation(model, tokenizer, prompt="The future of AI is", max_length=50):
    """Test text generation capability"""
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs.input_ids.long().to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_length=max_length,
            num_return_sequences=1,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text


## 6. Main Pruning Pipeline

In [16]:
def run_pruning_experiment(model_name, pruning_percentages=[10, 20, 30, 40, 50]):
    """Run complete pruning experiment for a model"""
    results = {
        'model_name': model_name,
        'pruning_results': {}
    }

    # Load evaluation dataset
    print("Loading evaluation dataset...")
    dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='validation')

    for percentage in pruning_percentages:
        print(f"\n{'='*60}")
        print(f"Testing {percentage}% pruning for {model_name}")
        print(f"{'='*60}")

        # Clear GPU memory before loading
        clear_gpu_memory()

        try:
            # Load model with 8-bit quantization
            model, tokenizer = load_model_8bit(model_name)

            # Initialize pruner
            pruner = LayerPruner(model, model_name)
            print(f"Model has {pruner.num_layers} layers")

            # Get pruning indices
            start_idx, end_idx = pruner.get_optimal_pruning_layers(percentage)

            if start_idx is not None:
                print(f"Pruning {percentage}%: layers {start_idx} to {end_idx}")

                # Evaluate before pruning
                print("\nEvaluating before pruning...")
                perplexity_before = evaluate_perplexity(model, tokenizer, dataset)
                generation_before = evaluate_generation(model, tokenizer)

                # Perform pruning
                pruner.prune_layers(start_idx, end_idx)
                get_gpu_memory_info()

                # Evaluate after pruning
                print("\nEvaluating after pruning...")
                perplexity_after = evaluate_perplexity(model, tokenizer, dataset)
                generation_after = evaluate_generation(model, tokenizer)

                # Store results
                results['pruning_results'][percentage] = {
                    'layers_pruned': f"{start_idx}-{end_idx}",
                    'perplexity_before': perplexity_before,
                    'perplexity_after': perplexity_after,
                    'perplexity_increase': (perplexity_after - perplexity_before) / perplexity_before * 100,
                    'generation_before': generation_before,
                    'generation_after': generation_after
                }

                print(f"\nResults for {percentage}% pruning:")
                print(f"Perplexity: {perplexity_before:.2f} → {perplexity_after:.2f} "
                      f"({results['pruning_results'][percentage]['perplexity_increase']:.1f}% increase)")
                print(f"Generation before: {generation_before}")
                print(f"Generation after: {generation_after}")

            # Clean up
            del model, tokenizer
            clear_gpu_memory()

        except Exception as e:
            print(f"Error testing {percentage}% pruning: {str(e)}")
            results['pruning_results'][percentage] = {'error': str(e)}

    return results

## 7. Results Visualization

In [17]:
def plot_pruning_results(all_results):
    """Create visualization of pruning results"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # Plot perplexity changes
    for model_results in all_results:
        model_name = model_results['model_name'].split('/')[-1]
        percentages = []
        perplexities = []

        for pct, res in model_results['pruning_results'].items():
            if 'error' not in res:
                percentages.append(pct)
                perplexities.append(res['perplexity_after'])

        ax1.plot(percentages, perplexities, marker='o', label=model_name)

    ax1.set_xlabel('Pruning Percentage (%)')
    ax1.set_ylabel('Perplexity')
    ax1.set_title('Perplexity vs Pruning Percentage')
    ax1.legend()
    ax1.grid(True)

    # Plot perplexity increase percentage
    for model_results in all_results:
        model_name = model_results['model_name'].split('/')[-1]
        percentages = []
        increases = []

        for pct, res in model_results['pruning_results'].items():
            if 'error' not in res:
                percentages.append(pct)
                increases.append(res['perplexity_increase'])

        ax2.plot(percentages, increases, marker='s', label=model_name)

    ax2.set_xlabel('Pruning Percentage (%)')
    ax2.set_ylabel('Perplexity Increase (%)')
    ax2.set_title('Relative Perplexity Increase')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig('pruning_results.png', dpi=300, bbox_inches='tight')
    plt.show()

## Main Execution

In [ ]:
def main():
    """Main execution function"""
    # Model configurations
    models_to_test = [
        {
            'name': 'Qwen/Qwen2.5-1.5B',
            'fallback': 'Qwen/Qwen2.5-0.5B'  # Smaller version if memory issues
        },
        {
            'name': 'Qwen/Qwen2-7B',
            'fallback': 'Qwen/Qwen2-1.5B'
        },
        {
            'name': 'meta-llama/Llama-3.2-3B',
            'fallback': 'meta-llama/Llama-3.2-1B'
        }
    ]

    all_results = []

    for model_config in models_to_test:
        print(f"\n{'#'*80}")
        print(f"Testing model: {model_config['name']}")
        print(f"{'#'*80}")

        try:
            # Try main model first
            results = run_pruning_experiment(
                model_config['name'],
                pruning_percentages=[10, 20, 30, 40, 50]
            )
            all_results.append(results)

        except Exception as e:
            print(f"Failed to test {model_config['name']}: {str(e)}")
            print(f"Trying fallback model: {model_config['fallback']}")

            try:
                # Try fallback model
                results = run_pruning_experiment(
                    model_config['fallback'],
                    pruning_percentages=[10, 20, 30, 40, 50]
                )
                all_results.append(results)
            except Exception as e2:
                print(f"Fallback also failed: {str(e2)}")

    # Save results
    with open('pruning_results.json', 'w') as f:
        json.dump(all_results, f, indent=2)

    # Create visualizations
    if all_results:
        plot_pruning_results(all_results)

    print("\n" + "="*80)
    print("Pruning experiments completed!")
    print("Results saved to pruning_results.json")
    print("="*80)

if __name__ == "__main__":
    main()


################################################################################
Testing model: Qwen/Qwen2.5-1.5B
################################################################################
Loading evaluation dataset...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/733k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/6.36M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]